# Модуль 2 · Урок 2 — Архітектура: `@property`, SOLID, сервіси, Repository

👋 В Уроці 1 ми опанували основи ООП. Тепер найцікавіше — як із цих цеглинок будують
**підтримуваний код у реальних проєктах**: `@property`, композиція, принципи **SOLID**,
**сервісний шар** і патерн **Repository**. Наприкінці розберемо, як виглядає продакшн-репозиторій,
лінтери та змінні середовища.

**Передумова:** [Урок 1 — основи ООП](lektsiya-1-osnovy-oop.ipynb)

> 🧩 До цього уроку йде **готовий приклад-проєкт**: [skelet-proyektu](skelet-proyektu/).
> Ми розберемо його по частинах, а весь код можна запустити.

## 1. `@property` — «розумні» атрибути

`@property` дозволяє звертатися до методу **як до атрибута** (без дужок). Це дає найкраще з двох
світів: зручний доступ `obj.value` **плюс** контроль (валідація, обчислення) під капотом.

Три типові застосування:
1. **Обчислюваний атрибут** — значення рахується на льоту.
2. **Валідація при присвоєнні** — сеттер перевіряє нове значення.
3. **Атрибут лише для читання** — є геттер, немає сеттера.

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):                 # обчислюваний атрибут — БЕЗ дужок при доступі
        return 3.14159 * self.radius ** 2

c = Circle(5)
print("Площа:", c.area)             # звертаємось як до атрибута: c.area, а не c.area()
c.radius = 10
print("Нова площа:", c.area)        # перерахувалось автоматично

In [ ]:
# Валідація через сеттер: не дамо задати від'ємну температуру нижче абсолютного нуля
class Temperature:
    def __init__(self, celsius=0):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Нижче абсолютного нуля не буває")
        self._celsius = value

    @property
    def fahrenheit(self):           # обчислюваний, лише для читання
        return self._celsius * 9 / 5 + 32

t = Temperature(25)
print(t.celsius, "C =", t.fahrenheit, "F")

t.celsius = 100                     # пройде через сеттер
print(t.celsius, "C =", t.fahrenheit, "F")

try:
    t.celsius = -300                # спрацює валідація
except ValueError as e:
    print("Помилка:", e)

## 2. Композиція vs наслідування

Крім наслідування («**є** різновидом», *is-a*), об'єкти можна поєднувати **композицією**
(«**має**», *has-a*): один об'єкт **містить** інші й делегує їм роботу.

- **Наслідування:** `Dog` **є** `Animal`.
- **Композиція:** `Car` **має** `Engine`.

> 🔎 **Важливо знати (питання співбесіди):** «**Prefer composition over inheritance**» —
> надавайте перевагу композиції. Глибокі ієрархії наслідування крихкі; композиція гнучкіша,
> бо ви збираєте поведінку з незалежних частин.

In [ ]:
class Engine:
    def start(self):
        return "двигун заведено"

class Car:
    def __init__(self, model):
        self.model = model
        self.engine = Engine()        # Car МАЄ Engine (композиція)

    def drive(self):
        return f"{self.model}: {self.engine.start()}, поїхали"

car = Car("Toyota")
print(car.drive())

## 3. 🔎 Важливо знати: принципи SOLID

**SOLID** — 5 принципів проєктування, які роблять код гнучким і підтримуваним. Їх **дуже часто**
питають на співбесідах. Розберемо кожен коротко.

- **S — Single Responsibility** (єдина відповідальність): клас має робити **одну** річ / мати
  **одну** причину для зміни. Не змішуйте, наприклад, бізнес-логіку й збереження у файл.
- **O — Open/Closed** (відкритість/закритість): код **відкритий для розширення**, але
  **закритий для змін** — нову поведінку додаємо новим класом, а не переписуючи наявний.
- **L — Liskov Substitution** (підстановка Лісков): об'єкт-нащадок має бути **взаємозамінним**
  з батьком без поломки програми.
- **I — Interface Segregation** (розділення інтерфейсів): краще кілька **вузьких** інтерфейсів,
  ніж один «товстий», що змушує реалізовувати непотрібне.
- **D — Dependency Inversion** (інверсія залежностей): залежте від **абстракцій**, а не від
  конкретних реалізацій.

Покажемо два найважливіші на практиці — **S** і **O/D**.

In [ ]:
# S — Single Responsibility.
# ❌ Один клас робить усе: і рахує, і зберігає, і форматує.
# ✅ Розділяємо відповідальності на окремі класи:
class Invoice:
    def __init__(self, amount):
        self.amount = amount

class InvoiceCalculator:
    def with_tax(self, invoice, rate=0.2):
        return invoice.amount * (1 + rate)

class InvoicePrinter:
    def to_text(self, invoice):
        return f"Рахунок на суму {invoice.amount} грн"

inv = Invoice(1000)
print(InvoiceCalculator().with_tax(inv))   # 1200.0
print(InvoicePrinter().to_text(inv))

In [ ]:
# O/D — Open/Closed + Dependency Inversion через абстракцію.
from abc import ABC, abstractmethod

class Notifier(ABC):                 # абстракція (інтерфейс)
    @abstractmethod
    def send(self, message): ...

class EmailNotifier(Notifier):
    def send(self, message):
        return f"EMAIL: {message}"

class SmsNotifier(Notifier):
    def send(self, message):
        return f"SMS: {message}"

# Функція залежить від АБСТРАКЦІЇ Notifier, а не від конкретного класу.
# Додати новий канал (напр. Telegram) можна БЕЗ зміни цього коду -> Open/Closed.
def notify_user(notifier: Notifier, message):
    return notifier.send(message)

print(notify_user(EmailNotifier(), "Вітаємо!"))
print(notify_user(SmsNotifier(), "Код: 1234"))

## 4. Розділення відповідальностей (separation of concerns)

Великий застосунок розбивають на **шари**, де кожен відповідає за своє. Це прямий наслідок
принципу Single Responsibility, але на рівні всього проєкту:

```
   ┌─────────────────────────────────────────────┐
   │  Домен (domain)        — бізнес-сутності      │  "ЩО це"
   ├─────────────────────────────────────────────┤
   │  Репозиторії (repos)   — доступ до даних/БД    │  "ДЕ зберігати"
   ├─────────────────────────────────────────────┤
   │  Сервіси (services)    — бізнес-логіка          │  "ЩО з цим робити"
   ├─────────────────────────────────────────────┤
   │  Точка входу (main)    — збирає все разом        │
   └─────────────────────────────────────────────┘
```

Переваги: кожен шар можна **розробляти, тестувати й змінювати окремо**. Наприклад, поміняти
базу даних — і жоден сервіс не постраждає.

## 5. Патерн Repository

**Repository** — шар, що **ховає деталі зберігання даних** за простим інтерфейсом
(`add`, `get`, `list`…). Решта коду не знає, чи це база даних, файл, чи пам'ять.

Це поєднання **абстракції** (Урок 1) та **інверсії залежностей** (SOLID). Побудуймо працюючий
приклад: абстрактний репозиторій + реалізація в пам'яті.

In [ ]:
from abc import ABC, abstractmethod

# 1) Доменна модель
class Task:
    def __init__(self, title, id=None, done=False):
        self.id = id
        self.title = title
        self.done = done
    def __repr__(self):
        mark = "✓" if self.done else " "
        return f"Task(id={self.id}, [{mark}] {self.title})"

# 2) Абстракція сховища (контракт)
class AbstractTaskRepository(ABC):
    @abstractmethod
    def add(self, task): ...
    @abstractmethod
    def get(self, task_id): ...
    @abstractmethod
    def list(self): ...

# 3) Конкретна реалізація — у пам'яті (у проді був би SQL)
class InMemoryTaskRepository(AbstractTaskRepository):
    def __init__(self):
        self._items = {}
        self._next_id = 1
    def add(self, task):
        if task.id is None:
            task.id = self._next_id
        self._items[task.id] = task
        self._next_id = max(self._next_id, task.id + 1)
        return task
    def get(self, task_id):
        return self._items.get(task_id)
    def list(self):
        return list(self._items.values())

repo = InMemoryTaskRepository()
repo.add(Task("Вивчити ООП"))
repo.add(Task("Написати проєкт"))
print(repo.list())
print("Задача 1:", repo.get(1))

## 6. Сервісний шар (service layer)

**Сервіс** містить **бізнес-логіку** й працює через репозиторій. Він **не знає**, як саме
зберігаються дані — отримує репозиторій ззовні (**впровадження залежностей**). Продовжимо
попередній приклад:

In [ ]:
class TaskNotFoundError(Exception):
    pass

class TaskService:
    def __init__(self, repository):        # залежимо від АБСТРАКЦІЇ репозиторію
        self._repo = repository

    def create_task(self, title):
        return self._repo.add(Task(title))

    def complete_task(self, task_id):
        task = self._repo.get(task_id)
        if task is None:
            raise TaskNotFoundError(f"Завдання {task_id} не знайдено")
        task.done = True
        return task

    def all_tasks(self):
        return self._repo.list()

# Збираємо разом: сервіс + конкретний репозиторій
service = TaskService(InMemoryTaskRepository())
service.create_task("Підготуватись до співбесіди")
t = service.create_task("Зробити домашку")
service.complete_task(t.id)

for task in service.all_tasks():
    print(task)

> 💡 Оскільки сервіс залежить від **абстрактного** репозиторію, для тестів можна підставити
> `InMemory`-версію (швидко, без БД), а в проді — реальну базу. Це і є сила такого поділу.

## 7. Як виглядають продакшн-репозиторії

У реальному проєкті код розкладають по файлах і теках приблизно так (**src-layout**):

```
skelet-proyektu/
├── pyproject.toml          # метадані проєкту + конфіг лінтера/тестів
├── .env.example            # приклад змінних середовища
├── .gitignore
├── src/
│   └── taskapp/
│       ├── config.py           # конфігурація з env
│       ├── exceptions.py       # власні винятки
│       ├── domain/models.py    # 🟢 сутності (Task)
│       ├── repositories/       # 🔵 доступ до даних
│       │   ├── base.py         #    AbstractTaskRepository
│       │   └── in_memory.py    #    InMemoryTaskRepository
│       ├── services/           # 🟣 бізнес-логіка
│       │   └── task_service.py
│       └── main.py             # точка входу (composition root)
└── tests/
    └── test_task_service.py
```

**Саме це** ми щойно написали в комірках — тільки розкладене по файлах. Готовий приклад лежить
поруч: [skelet-proyektu](skelet-proyektu/) (див. його [README](skelet-proyektu/README.md)).

Запуск і тести:
```bash
cd skelet-proyektu
PYTHONPATH=src python -m taskapp.main          # запустити застосунок
PYTHONPATH=src python tests/test_task_service.py   # тести без сторонніх бібліотек
```

Що робить проєкт «продакшн-подібним»:
- **чіткі шари** (domain / repositories / services) — легко тестувати й змінювати;
- **тести** поруч із кодом;
- **`pyproject.toml`** з метаданими та конфігом інструментів;
- **конфіг через env**, а не хардкод;
- **лінтер** для єдиного стилю (нижче).

## 8. Лінтери та форматери

**Лінтер** автоматично знаходить проблеми в коді (невикористані змінні, погані імпорти,
порушення стилю), а **форматер** приводить код до єдиного вигляду. Це стандарт у будь-якій команді.

Популярні інструменти:
- **Ruff** — дуже швидкий лінтер + форматер (замінює Flake8, isort, частину Black). Сучасний вибір.
- **Black** — форматер «без суперечок» (єдиний стиль).
- **mypy** — перевірка типів (за анотаціями).

Налаштування зазвичай кладуть у `pyproject.toml` (як у нашому скелеті):
```toml
[tool.ruff]
line-length = 100
target-version = "py310"

[tool.ruff.lint]
select = ["E", "F", "I", "UP", "B"]   # стиль, помилки, порядок імпортів, сучасний синтаксис, баги
```

Використання:
```bash
pip install ruff
ruff check .     # знайти проблеми
ruff format .    # відформатувати код
```

## 9. Змінні середовища (`.env`) та конфігурація

**Правило:** налаштування (адреси БД, ключі, паролі) **не хардкодять** у коді й **не комітять**
у git. Їх передають через **змінні середовища** — це принцип **12-factor app**.

- Реальні значення кладуть у файл **`.env`** (він у `.gitignore`, тобто **не** потрапляє в git).
- У репозиторії лишають **`.env.example`** — шаблон без секретів.
- Код читає значення через `os.getenv(...)` (або бібліотеку `python-dotenv`, `pydantic-settings`).

In [ ]:
import os

# Читаємо налаштування з середовища; якщо змінної немає — беремо значення за замовчуванням.
app_name = os.getenv("APP_NAME", "taskapp")
debug = os.getenv("DEBUG", "false").lower() == "true"
database_url = os.getenv("DATABASE_URL", "memory://")

print("app_name:", app_name)
print("debug:", debug)
print("database_url:", database_url)

У нашому скелеті це оформлено класом (файл `config.py`) — зверніть увагу на `@classmethod`
як «фабрику» налаштувань:
```python
@dataclass(frozen=True)
class Settings:
    app_name: str
    debug: bool
    database_url: str

    @classmethod
    def from_env(cls):
        return cls(
            app_name=os.getenv("APP_NAME", "taskapp"),
            debug=os.getenv("DEBUG", "false").lower() == "true",
            database_url=os.getenv("DATABASE_URL", "memory://"),
        )

settings = Settings.from_env()
```

> ⚠️ **Ніколи** не комітьте `.env` із реальними паролями/ключами. Витік секретів у git —
> класична (і дорога) помилка.

# ✅ Підсумок уроку

- **`@property`** — доступ до методу як до атрибута: обчислювані значення, валідація, read-only.
- **Композиція** («має») часто краща за наслідування («є») — гнучкіше.
- **SOLID** — 5 принципів гнучкого коду; ключові на практиці: Single Responsibility та
  Dependency Inversion (залежати від абстракцій).
- **Розділення відповідальностей** — код по шарах: домен / репозиторії / сервіси.
- **Repository** ховає сховище за інтерфейсом; **сервіс** містить бізнес-логіку.
- **Продакшн-структура:** src-layout, тести, `pyproject.toml`, конфіг через env.
- **Лінтери** (ruff) тримають стиль; **env** зберігають налаштування поза кодом.

### 📝 Домашнє завдання
[domashnie-zavdannia-lektsiya-2.ipynb](domashnie-zavdannia-lektsiya-2.ipynb)

### 🎉 Вітаємо із завершенням курсу!
Ви пройшли шлях від `print("Привіт")` до архітектури проєкту з патернами. Далі — **практика**:
візьміть [skelet-proyektu](skelet-proyektu/) за основу й розширюйте його.

### 📚 Хочу знати більше
- SOLID простими словами: <https://realpython.com/solid-principles-python/>
- Repository pattern (Architecture Patterns with Python): <https://www.cosmicpython.com/>
- Ruff (лінтер): <https://docs.astral.sh/ruff/>
- 12-Factor App (конфігурація): <https://12factor.net/config>
- `pydantic-settings` (конфіг із env): <https://docs.pydantic.dev/latest/concepts/pydantic_settings/>